[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-06.ipynb)

# 6주차 실습: 멀티모달 표현의 결합

**목표.** 제주 공개 사진 5장으로 두 가지를 해 본다.

1. **CLIP 제로샷 분류**: 사진과 후보 문장의 유사도 히트맵을 그리고, 라벨 언어(영어/한국어)가 결과를 어떻게 바꾸는지 본다
2. **SmolVLM 질의응답**: 사진을 언어모형에 넣고 질문해 답을 얻는다. **사진만 바꿔** 답이 사진을 따라 움직이는지(근거), 한국어 질문이 어떻게 깨지는지(언어 편향) 확인한다

GPU는 필요 없다. 노트북이 실행 시점에 사진과 모델을 내려받는다. 사진은 위키미디어 공용과 공공누리의 제주 공개 사진이고, 출처와 라이선스는 강의 노트 6주차 [사진 출처] 표에 있다.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

> transformers가 5.x면 시각 모델 클래스가 `AutoModelForImageTextToText`다. 4.x의 `AutoModelForVision2Seq`는 이름이 바뀌었다. 이 노트북은 5.x 클래스를 쓴다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers torch torchvision pillow

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 제주 공개 사진 5장 내려받기

위키미디어 공용(및 공공누리)에서 라이선스가 확인된 제주 사진 5장을 내려받는다. 강의 노트의 실측과 같은 사진이다.

In [ ]:
import urllib.request
import time

IMAGES = {
    # 파일명: (다운로드 주소, 위키미디어 공용 원본 제목)
    "seongsan-sunrise": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f4/Sunrise_with_Seongsan_Ilchulbong.jpg/960px-Sunrise_with_Seongsan_Ilchulbong.jpg",
        "Sunrise with Seongsan Ilchulbong",
    ),
    "seongsan-carpark": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b0/Sunrise_peak_carpark_-_panoramio.jpg/960px-Sunrise_peak_carpark_-_panoramio.jpg",
        "Sunrise peak carpark - panoramio",
    ),
    "hallasan-yeongsil": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e5/Wooden_staircase_along_Yeongsil_Trail_with_the_mountains_of_Hallasan_Park_Jeju_Island_South_Korea.jpg/960px-Wooden_staircase_along_Yeongsil_Trail_with_the_mountains_of_Hallasan_Park_Jeju_Island_South_Korea.jpg",
        "Wooden staircase along Yeongsil Trail",
    ),
    "manjanggul": (
        "https://upload.wikimedia.org/wikipedia/commons/8/80/Lava_Manjanggul_Cave.jpg",
        "Lava Manjanggul Cave",
    ),
    "udo": (
        "https://upload.wikimedia.org/wikipedia/commons/thumb/0/04/Udo%2C_Jeju_Province%2C_South_Korea_02.jpg/960px-Udo%2C_Jeju_Province%2C_South_Korea_02.jpg",
        "Udo, Jeju Province, South Korea 02",
    ),
}

import os
os.makedirs("week06-imgs", exist_ok=True)

def fetch(url, tries=4):
    # 위키미디어는 짧은 시간에 요청이 몰리면 429를 돌려준다. 기다렸다 다시 요청한다.
    for k in range(tries):
        req = urllib.request.Request(url, headers={"User-Agent": "deepnlp-course/1.0"})
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 429 and k < tries - 1:
                wait = 15 * (k + 1)
                print(f"  429, {wait}초 후 재시도 ({k + 1}/{tries - 1})")
                time.sleep(wait)
            else:
                raise

for name, (url, _title) in IMAGES.items():
    path = f"week06-imgs/{name}.jpg"
    if not os.path.exists(path):
        data = fetch(url)
        with open(path, "wb") as f:
            f.write(data)
    print(f"{path}: {os.path.getsize(path) // 1024} KB")

### 1-2. 사진 한눈에 보기

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (name, (_url, title)) in zip(axes, IMAGES.items()):
    img = Image.open(f"week06-imgs/{name}.jpg").convert("RGB")
    ax.imshow(img)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1-3. CLIP 제로샷 - 사진과 문장의 유사도

CLIP은 이미지 인코더와 텍스트 인코더 **둘**을 가진다. 두 인코더가 각자 벡터를 만들고, 벡터를 길이 1로 정규화한 뒤 내적하면 코사인 유사도가 나온다.

사진 5장 x 후보 문장 5개의 유사도 행렬을 만들어 본다. 각 행에서 가장 큰 값이 모델이 고른 문장이다.

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print(f"logit_scale: {clip.logit_scale.exp().item():.1f}")

def clip_similarity(image_paths, texts):
    # 이미지와 문장을 각자 인코딩해 정규화한 뒤 내적한다
    imgs = [Image.open(p).convert("RGB") for p in image_paths]
    with torch.no_grad():
        img_inputs = clip_proc(images=imgs, return_tensors="pt")
        img_emb = clip.get_image_features(pixel_values=img_inputs["pixel_values"])
        if hasattr(img_emb, "pooler_output"):
            img_emb = img_emb.pooler_output
        txt_inputs = clip_proc(text=texts, return_tensors="pt", padding=True)
        txt_emb = clip.get_text_features(
            input_ids=txt_inputs["input_ids"], attention_mask=txt_inputs["attention_mask"]
        )
        if hasattr(txt_emb, "pooler_output"):
            txt_emb = txt_emb.pooler_output
        img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
        return (img_emb @ txt_emb.T).tolist()

# 동작 확인: 첫 사진 1장, 짧은 영어 라벨 5개
labels_en_coarse = [
    "a photo of a sunrise peak",
    "a photo of a parking lot",
    "a photo of a hiking trail",
    "a photo of a lava tube cave",
    "a photo of a coastal island",
]
paths = [f"week06-imgs/{name}.jpg" for name in IMAGES]
sim = clip_similarity(paths, labels_en_coarse)
print(f"\n{IMAGES['seongsan-sunrise'][1]} 행:")
for lab, v in zip(labels_en_coarse, sim[0]):
    print(f"  {v:+.3f}  {lab}")

### 1-4. 라벨 언어가 결과를 바꾼다 - 유사도 히트맵

같은 사진 5장에 라벨 조합을 네 가지로 바꿔가며 제로샷 분류를 돌린다.

- **영어 (짧은)**: `a photo of a ...` 뒤에 짧은 영어 라벨
- **영어 (장면 묘사)**: 장면을 묘사하는 긴 영어 라벨
- **한국어**: `... 사진` 형식의 한국어 라벨
- **한국어 + 영어 섞기**: 한국어 라벨에 영어를 괄호로 붙임

히트맵에서 행: 사진, 열: 후보 라벨이다. 가장 진한 칸이 모델이 고른 라벨이고, **대각선이 진하면 정답을 골라낸 것**이다.

In [ ]:
LABEL_SETS = {
    "en_coarse": labels_en_coarse,
    "en_scenario": [
        "a photo of a sunrise over a volcanic crater by the sea",
        "a photo of a parking lot full of tour buses",
        "a photo of a wooden staircase on a forest hiking trail",
        "a photo of a dark lava tube cave interior",
        "a photo of green sea cliffs and a small island",
    ],
    "ko": [
        "성산일출봉 사진",
        "주차장 사진",
        "등산로 사진",
        "용암동굴 사진",
        "해안 섬 사진",
    ],
    "ko_english_mix": [
        "성산일출봉 (Seongsan Ilchulbong) sunrise peak photo",
        "주차장 (parking lot) photo",
        "등산로 (hiking trail) photo",
        "용암동굴 (lava tube cave) photo",
        "해안 섬 (coastal island) photo",
    ],
}

img_names = list(IMAGES)
all_sims = {}
for set_name, labels in LABEL_SETS.items():
    all_sims[set_name] = clip_similarity(paths, labels)
    correct = sum(1 for i in range(len(paths)) if max(range(len(labels)), key=lambda j: all_sims[set_name][i][j]) == i)
    print(f"{set_name}: 대각선 정답 {correct}/{len(paths)}")

fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
short = {"en_coarse": "EN short", "en_scenario": "EN scenario", "ko": "KO", "ko_english_mix": "KO+EN mix"}
for ax, (set_name, labels) in zip(axes, LABEL_SETS.items()):
    data = all_sims[set_name]
    ax.imshow(data, cmap="viridis", vmin=0.1, vmax=0.35)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(range(1, len(labels) + 1), fontsize=8)
    ax.set_yticks(range(len(img_names)))
    ax.set_yticklabels(img_names, fontsize=8)
    for i in range(len(paths)):
        for j in range(len(labels)):
            ax.text(j, i, f"{data[i][j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if data[i][j] < 0.3 else "black")
    ax.set_title(short[set_name], fontsize=10)
plt.suptitle("CLIP zero-shot cosine similarity (rows: images, cols: labels)", fontsize=11)
plt.tight_layout()
plt.show()

print("열 라벨 번호:")
for set_name, labels in LABEL_SETS.items():
    print(f"  {set_name}: " + ", ".join(f"{j+1}={l}" for j, l in enumerate(labels)))

### 1-5. 한국어 라벨은 토큰에서 어떻게 되는가

한국어 라벨이 무너지는 원인을 토큰에서 확인한다. CLIP의 토크나이저는 한국어 글자를 **바이트 조각**으로 자른다. 토큰 자체는 만들어지므로 무너지는 원인은 토큰화가 아니라, 그 토큰들이 정렬 공간에서 사진 방향과 이어진 적이 없다는 것(학습 데이터가 영어 위주)이다.

In [ ]:
ko_text = "성산일출봉 사진"
ko_inputs = clip_proc(text=[ko_text], return_tensors="pt")
ko_ids = ko_inputs["input_ids"][0].tolist()
en_text = "a photo of a sunrise peak"
en_inputs = clip_proc(text=[en_text], return_tensors="pt")
en_ids = en_inputs["input_ids"][0].tolist()

print(f"한국어 '{ko_text}' -> 토큰 {len(ko_ids)}개: {ko_ids}")
print(f"영어   '{en_text}' -> 토큰 {len(en_ids)}개: {en_ids}")
print(f"어휘 크기: {clip.config.text_config.vocab_size} (49408 = BPE 바이트 어휘)")
print()
print("한국어는 글자가 바이트 조각으로 분해된다. 토큰은 만들어지지만,")
print("그 토큰들이 사진 방향과 정렬된 적이 없으므로 유사도에 방향 차이가 생기지 않는다.")

### 1-6. SmolVLM 질의응답 - 사진을 언어모형에 넣기

이제 생성형이다. SmolVLM-256M-Instruct는 사진을 시각 토큰으로 바꿔 언어모형의 문맥에 끼워 넣는다. 사진 1장이 **832개의 시각 토큰**을 쓴다(960x640 기준).

절차는 세 단계다.

1. 채팅 템플릿으로 대화 형식 프롬프트를 조립한다 (이미지 자리는 표시자로 둔다)
2. 사진과 프롬프트를 프로세서에 넣어 모델 입력을 만든다
3. `generate()`로 답을 생성한다

입력 토큰 수에서 시각 토큰이 차지하는 비중도 함께 센다.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

vlm_name = "HuggingFaceTB/SmolVLM-256M-Instruct"
vlm_processor = AutoProcessor.from_pretrained(vlm_name)
vlm = AutoModelForImageTextToText.from_pretrained(vlm_name).eval()

total_params = sum(p.numel() for p in vlm.parameters())
print(f"전체 파라미터: {total_params:,}")

def vlm_ask(image_path, question, max_new_tokens=48):
    # 사진과 질문을 채팅 템플릿에 넣어 답을 생성한다
    img = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_processor(images=img, text=prompt, return_tensors="pt")
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = vlm_processor.batch_decode(out, skip_special_tokens=True)[0]
    return text.split("Assistant:")[-1].strip()

def visual_token_count(image_path):
    # 시각 토큰(이미지 표시자)이 문맥에서 차지하는 수를 센다
    img = Image.open(image_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe this image."}]}]
    prompt = vlm_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_processor(images=img, text=prompt, return_tensors="pt")
    n_total = inputs["input_ids"].shape[1]
    img_token_id = getattr(vlm_processor, "image_token_id", None)
    n_vis = sum(1 for t in inputs["input_ids"][0].tolist() if t == img_token_id) if img_token_id is not None else -1
    return n_total, n_vis

first = f"week06-imgs/seongsan-sunrise.jpg"
n_total, n_vis = visual_token_count(first)
print(f"\n입력 토큰 {n_total}개 중 시각 토큰 {n_vis}개 ({n_vis / n_total:.0%})")
print("-> 사진이 문맥의 대부분을 차지한다 (문맥 잠식)")

print("\n동작 확인:")
print("Q: Describe this image in one sentence.")
print(f"A: {vlm_ask(first, 'Describe this image in one sentence.')}")

### 1-7. 사진만 바꿔 보기 - 답이 사진을 따라 움직이는가

질문을 고정하고 **사진만** 다섯 장으로 바꿔 본다. 답이 사진 내용을 따라 바뀌면 답이 이미지에 근거(grounded) 있다는 뜻이다.

In [ ]:
QUESTION_EN = "Describe this image in one sentence."
print(f"질문(고정): {QUESTION_EN}\n")
for name in IMAGES:
    path = f"week06-imgs/{name}.jpg"
    answer = vlm_ask(path, QUESTION_EN)
    print(f"[{name}]")
    print(f"  {answer}\n")

### 1-8. 한국어 질문은 어떻게 깨지는가

같은 사진에 한국어 질문을 넣어 본다. 강의 노트의 실측과 같이 대부분 실패한다. 질문을 되풀이하거나, 깨진 한글이 나오거나, 영어 답이 나온다.

원인은 모델 크기가 아니라 **학습 데이터의 언어 분포**다. 256M 모델의 지시 학습 데이터는 영어 위주라 한국어 지시의 다음 토큰 확률이 무너진다.

In [ ]:
QUESTIONS_KO = [
    "이 이미지를 한 문장으로 묘사하라.",
    "여기가 어디인가? 주된 피사체가 무엇인가?",
]
first = "week06-imgs/seongsan-sunrise.jpg"
for q in QUESTIONS_KO:
    print(f"Q: {q}")
    print(f"A: {vlm_ask(first, q)}\n")

print("비교: 같은 질문 1번을 영어로 넣으면")
print(f"A: {vlm_ask(first, 'Describe this image in one sentence.')}")

## 2. 한 지점만 바꿔 보기 - 입력 이미지를 바꾼다

아래 셀의 `# TODO`로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 규칙: `MY_IMAGE`를 다른 사진 이름으로 바꾼다. 후보는 `seongsan-sunrise`, `seongsan-carpark`, `hallasan-yeongsil`, `manjanggul`, `udo` 다섯 가지다. 바꾼 뒤 두 가지를 기록한다. (1) 같은 질문인데 답이 사진을 따라 바뀌는가. (2) 바꾼 사진의 특징이 답에 어떻게 반영되는가(또는 어두운 사진에서 답이 어떻게 무너지는가). 이것이 학습활동 [구현]의 기록이다.

In [ ]:
# TODO: 사진 이름을 바꾼다. 후보: seongsan-sunrise / seongsan-carpark / hallasan-yeongsil / manjanggul / udo
MY_IMAGE = "seongsan-sunrise"

# 아래는 그대로 둡니다
path = f"week06-imgs/{MY_IMAGE}.jpg"
print(f"사진: {MY_IMAGE}")
print(f"\n[영어 질문]")
q = "Describe this image in one sentence."
a = vlm_ask(path, q)
print(f"  Q: {q}")
print(f"  A: {a}")
print(f"\n[한국어 질문]")
q_ko = "이 이미지를 한 문장으로 묘사하라."
a_ko = vlm_ask(path, q_ko)
print(f"  Q: {q_ko}")
print(f"  A: {a_ko}")
print(f"\n[제로샷 - 이 사진은 다섯 라벨 중 어디에 가까운가]")
sim_one = clip_similarity([path], LABEL_SETS["en_coarse"])[0]
best = max(range(5), key=lambda j: sim_one[j])
for j, (lab, v) in enumerate(zip(LABEL_SETS["en_coarse"], sim_one)):
    mark = "  <-- 모델의 선택" if j == best else ""
    print(f"  {v:+.3f}  {lab}{mark}")

## 3. 정리 그래프

네 라벨 조합의 대각선 정답 수를 막대로 그린다. 라벨 언어가 제로샷을 어떻게 바꾸는지 한눈에 본다.

In [ ]:
accs = {}
for set_name, labels in LABEL_SETS.items():
    accs[set_name] = sum(
        1 for i in range(len(paths)) if max(range(len(labels)), key=lambda j: all_sims[set_name][i][j]) == i
    ) / len(paths)

plt.figure(figsize=(7, 4))
names = [short[n] for n in LABEL_SETS]
vals = [accs[n] for n in LABEL_SETS]
bars = plt.bar(names, vals, color=["#4263eb", "#2f9e44", "#e03131", "#f08c00"])
for bar, v in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width() / 2, v + 0.03, f"{v:.0%}", ha="center", fontsize=11)
plt.ylim(0, 1.15)
plt.ylabel("zero-shot diagonal accuracy")
plt.title("Label language changes CLIP zero-shot accuracy (5 Jeju photos)")
plt.tight_layout()
plt.show()

## 4. 확인 질문

1. 라벨 조합 네 가지의 대각선 정답 수는 각각 몇이었나요? 어느 조합이 가장 좋았나요?
2. 한국어 라벨의 히트맵은 어떤 모양이었나요? 그 모양이 말해 주는 것은 무엇인가요?
3. 1-7에서 사진을 바꿨을 때 답이 사진을 따라 바뀌었나요? 어느 사진에서 답이 가장 무너졌나요, 왜 그랬을까요?
4. 시각 토큰은 입력 토큰 몇 개 중 몇 개였나요? 사진을 더 크게 넣으면 이 수는 어떻게 될까요?
5. 한국어 질문의 답은 어떤 모양이었나요? 질문 언어와 답 언어의 관계에서 관찰한 것을 한 문장으로 써 보세요.

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 5. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-06/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.